# grad-expressed-in-out — worked example 1: ReLU backward via cached output indicator

> Worked example from [Delta Drills](https://delta-drills.vercel.app). Atom: `grad-expressed-in-out`.

**This is a worked example — read it, run each cell, and follow the reasoning.** It's study material, so there's nothing to submit here. Delta Drills hands you a hands-on version to complete yourself as you get comfortable with the idea.

## Setup

In [ ]:
import numpy as np
import torch as t
from torch import Tensor
import einops
from einops import rearrange, reduce, repeat

t.manual_seed(0)
np.random.seed(0)
# === manual autograd primitives — shared across all drills in this folder ===
from dataclasses import dataclass, field
from typing import Any, Callable, Optional

grad_tracking_enabled = True

@dataclass
class Recipe:
    func: Optional[Callable] = None
    args: tuple = ()
    kwargs: dict = field(default_factory=dict)
    parents: dict = field(default_factory=dict)

class MiniTensor:
    """A minimal Tensor wrapper for the ARENA-style manual-autograd drills.
    Wraps a raw `torch.Tensor` in `.array`. Carries an optional `.recipe`
    populated by wrap_forward_fn. `requires_grad` is set by the wrapper.
    `.grad` accumulates the leaf gradient at the end of the reverse pass."""
    def __init__(self, array, requires_grad: bool = False, recipe=None):
        self.array = array
        self.requires_grad = requires_grad
        self.recipe = recipe
        self.grad = None
    def __repr__(self):
        return f'MiniTensor({self.array!r}, requires_grad={self.requires_grad})'

## Concept

_First time on this topic? Run the **Setup** cell above and skim it: every class and helper mentioned below is defined there. You don't need to have done any other drill first._

ReLU's backward pass is a gate: the gradient flows through only where the forward output was positive. Expressed via the cached output `out`, the local Jacobian is simply `(out > 0).float()` — no need to compare against zero again using the original input `x`. This is the 'grad expressed in out' pattern: storing the forward output makes the backward computation cheaper or simpler than going back to the raw input.

## Worked solution

**Step 1 — understand the derivative.** `out = relu(x) = max(0, x)`. The derivative is 1 where `x > 0` and 0 elsewhere. But since `out` is already 0 where `x ≤ 0` (ReLU zeroed it), we can write the same condition as `out > 0`.

**Step 2 — form the gradient indicator.** `(out > 0).float()` produces a tensor of the same shape as `out`, with 1.0 at active positions and 0.0 at inactive ones. This is the local Jacobian diagonal.

**Step 3 — apply the chain rule.** `dL/dx = grad_out * (out > 0).float()`. We multiply elementwise — the gate passes upstream gradients through at active positions and blocks them at zero positions.

**Step 4 — verify using x.** We also compute the derivative directly from `x` and confirm both forms agree, illustrating that `out` is the convenient form but both are equivalent.

In [ ]:
import torch as t

t.manual_seed(0)

def relu_back(grad_out: t.Tensor, out: t.Tensor, x: t.Tensor) -> t.Tensor:
    # out > 0 is equivalent to x > 0 (ReLU zeros the negative part)
    # Using `out` avoids needing `x` at all during the backward pass.
    return grad_out * (out > 0).float()

# Exercise
t.manual_seed(0)
x = t.randn(4)
out = t.relu(x)
grad_out = t.ones_like(x)

result = relu_back(grad_out, out, x)

# Verify via x directly
expected = (x > 0).float()
assert t.allclose(result, expected), f"Expected {expected}, got {result}"
print(f"x:        {x.tolist()}")
print(f"out:      {out.tolist()}")
print(f"relu_back:{result.tolist()}")
print("Checked: relu_back via out matches derivative via x.")